In [1]:
model_name = "vit-ragdoll"


import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n



BENCHMARK_REPEAT=33
df = pd.DataFrame()

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

In [2]:
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * 0.23421 for k, v in model.state_dict().items()})
model = torch.compile(model, backend="inductor")
torch_results = []
for bs in range(1, 27):
    print("measuring #", bs)
    model = models.vit_b_16().train(False)
    model.load_state_dict({k: torch.ones_like(v) * 0.23421 for k, v in model.state_dict().items()})
    model = torch.compile(model, backend="inductor")
    device = torch.device('cpu:0')
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)
    """
    baseline_f = torch_model_benchmark(model, 
                             [image], 
                             device='cpu',
                             warmups=0,
                             repetitions=5, 
                             measure_count=5)
    baseline_f = np.mean(baseline_f)
    """
    baseline_b = torch_model_benchmark(torch.autograd.grad, 
                             [output, [image], grad], 
                             device='cpu',
                             warmups=0,
                             repetitions=11, 
                             measure_count=11)
    baseline_b = np.mean(baseline_b)
    print(baseline_b)
    #df = pd.concat([df, get_dataframe(baseline_f, baseline_b, "Torch Dynamo at batch-size = {}".format(bs))])
    #print(df)
    torch_results.append(baseline_b)

measuring # 1
75.98628635097724
measuring # 2


Process ForkProcess-27:
Process ForkProcess-32:
Process ForkProcess-31:
Process ForkProcess-2:
Process ForkProcess-3:
Process ForkProcess-21:
Process ForkProcess-6:
Process ForkProcess-30:
Process ForkProcess-7:
Process ForkProcess-10:
Process ForkProcess-29:
Process ForkProcess-12:
Process ForkProcess-4:
Process ForkProcess-19:
Process ForkProcess-8:
Process ForkProcess-11:
Process ForkProcess-25:
Process ForkProcess-17:
Process ForkProcess-16:
Process ForkProcess-5:
Process ForkProcess-18:
Process ForkProcess-1:
Process ForkProcess-9:
Process ForkProcess-28:
Process ForkProcess-15:
Process ForkProcess-13:
Process ForkProcess-14:
Process ForkProcess-20:
Process ForkProcess-26:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last)

In [ ]:
import gc
model_file_base = "vit.mlir"
strategy = "heuristic"
ragdoll_results = []
for bs in list(range(1, 36)):
    model_file = model_file_base + ".bs{}".format(bs)
    source_file = model_file + ".{}".format(strategy)
    target_file = source_file + ".vmfb"
    #"""
    # gen model with specified batch-size
    !ragdoll-opt {model_file_base} --ragdoll-autodiff-prepare-batch-size=batchsize={bs} > {model_file}
    
    !ragdoll-opt {model_file}  \
    --canonicalize \
    --enable-cse-in-legalizer \
    --symbol-dce \
    --ragdoll-autodiff-vjp-public-functions='strategy=heuristic' \
    --ragdoll-autodiff-vjp \
    --inline \
    --ragdoll-autodiff-inline-function-call \
    --ragdoll-initialisation \
    --eliminate-empty-tensors \
    --ragdoll-legalise-to-iree-compatibility \
    --ragdoll-raise-linalg-to-tosa \
    --ragdoll-forward-func-removal \
    --canonicalize \
    --cse > {source_file}
    
    !iree-compile {source_file} \
    -o {target_file} \
    --iree-hal-target-backends=llvm-cpu \
    --iree-llvmcpu-fail-on-out-of-bounds-stack-allocation=0 \
    --iree-opt-const-eval=1 \
    --iree-opt-const-expr-hoisting=1 \
    --iree-opt-numeric-precision-reduction=1 \
    --iree-llvmcpu-target-cpu-features=host \
    --iree-llvmcpu-enable-ukernels=all \
    --iree-llvmcpu-slp-vectorization=1 \
    --iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
    --iree-llvmcpu-target-triple=x86_64-pc-linux-elf
    #"""
    
    #ragdoll_binary = load_executable(target_file)

    image = torch.randn(bs, 3, 224, 224)
    image_np = image.detach().cpu().numpy()
    image_t = torch.randn(bs, 224, 224, 3)
    image_np_t = image_t.detach().cpu().numpy()
    model = models.vit_b_16().train(False)
    model.load_state_dict({k: torch.ones_like(v) * 0.23421 for k, v in model.state_dict().items()})
    output = model(image)
    grad = torch.randn_like(output)
    grad_np = grad.cpu().numpy()

    #try:
        #f1 = timeit("ragdoll_binary.forward(image_np_t)") / BENCHMARK_REPEAT
        #print('ragdoll-opt1-gpu-forward in timeit: ', f1)
    #"""
    b1 = ragdoll_model_benchmark(
        target_file,
        "dforward",
        [(bs, 1000)],
        device='cpu',
        warmups=3,
        repetitions=BENCHMARK_REPEAT, 
        measure_count=1)
    b1 = np.mean(b1)
    """
    b1 = timeit("ragdoll_binary.dforward(grad_np)") / BENCHMARK_REPEAT
    """
    print('ragdoll-opt1-gpu-forward in timeit: ', b1)
    ragdoll_results.append(b1)


In [ ]:
df.style.hide(axis="index")
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")